# Training Baseline

Notebook base para entrenar modelos individuales reutilizando utilidades en `src/`.

Secciones: configuración, carga de datos, datasets/dataloaders, modelo, pérdida/optimizador, entrenamiento, evaluación y Grad-CAM.

## 1) Imports y configuración básica

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch

# Asegurar que la raíz del repo está en sys.path (notebook ubicado en `baseline/`)
sys.path.append(str(Path('..').resolve()))

# Importar utilidades desde src
from src import (
    set_global_seed,
    get_device,
    CONFIG,
    CATALOG_PATH,
    FITS_DIR,
    GalaxyDataset,
    build_resnet18,
    compute_classification_metrics,
    train_loop,
    generate_gradcam_visualization,
)

print('Imports OK')

## 2) Semilla y dispositivo

In [ ]:
# Config reproducibilidad y dispositivo
set_global_seed(CONFIG['seed'])
device = get_device()
print(f'Using device: {device}')

## 3) Carga del catálogo y división stratificada

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

catalog_path = Path(CATALOG_PATH)
assert catalog_path.exists(), f'Catalog not found: {catalog_path}'
df = pd.read_csv(catalog_path)
df.drop(columns=['source'], errors='ignore', inplace=True)

# Excluir registros corruptos si existe reporte procesado
report_path = Path('..') / 'data' / 'processed' / 'dataset_validation_report.csv'
if report_path.exists():
    report_df = pd.read_csv(report_path)
    invalid_ids = report_df[report_df['complete'] == False]['name'].tolist()
    df = df[~df['name'].isin(invalid_ids)].reset_index(drop=True)

# Stratified split: 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=CONFIG['seed'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=CONFIG['seed'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 4) Datasets y DataLoaders

In [ ]:
from torch.utils.data import DataLoader

train_dataset = GalaxyDataset(train_df, FITS_DIR, target_shape=CONFIG['image_size'], augment=True)
val_dataset = GalaxyDataset(val_df, FITS_DIR, target_shape=CONFIG['image_size'], augment=False)
test_dataset = GalaxyDataset(test_df, FITS_DIR, target_shape=CONFIG['image_size'], augment=False)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])

print('Datasets and DataLoaders ready')

## 5) Construcción del modelo (ResNet18 baseline)

In [ ]:
# Construir ResNet18 con las utilidades de src.models
model, trainable_params, frozen_params = build_resnet18(num_classes=CONFIG['num_classes'], pretrained=True, freeze_backbone=CONFIG.get('freeze_backbone', True), device=device)
print(f'Trainable params: {trainable_params:,} | Frozen params: {frozen_params:,}')

# Verificación rápida de forward con un batch (si hay datos suficientes)
try:
    sample_images, _ = next(iter(train_loader))
    sample_images = sample_images.to(device)
    with torch.no_grad():
        outputs = model(sample_images[:2])
    print('Forward OK, outputs shape:', outputs.shape)
except Exception as e:
    print('Forward test skipped:', e)

## 6) Pérdida, pesos de clase y optimizador

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Calcular pesos de clase a partir de train_df
from collections import Counter
counts = Counter(train_df['label'].tolist())
num_samples = len(train_df)
num_classes = CONFIG['num_classes']
class_weights = [num_samples / (num_classes * counts.get(i, 1)) for i in range(num_classes)]
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])

print('Criterion and optimizer ready')

## 7) Entrenamiento

In [ ]:
# Ejecutar loop de entrenamiento definido en src.train
model, history = train_loop(model, train_loader, val_loader, criterion, optimizer, device, epochs=1)

# Guardar checkpoint simple
ckpt_path = Path('training_baseline_checkpoint.pth')
torch.save({'model_state_dict': model.state_dict(), 'history': history}, ckpt_path)
print(f'Checkpoint saved: {ckpt_path}')

## 8) Evaluación final en Test Set

In [ ]:
# Inferencia sobre test set
model.eval()
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_probs = np.array(all_probs)

metrics = compute_classification_metrics(y_true, y_pred, y_probs)
print('Test metrics:')
for k, v in metrics.items():
    print(f'{k}: {v}')

In [ ]:
# Mostrar curvas de aprendizaje
epochs_range = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(epochs_range, history['train_loss'], label='train_loss')
plt.plot(epochs_range, history['val_loss'], label='val_loss')
plt.legend()
plt.title('Loss')

plt.subplot(1,2,2)
plt.plot(epochs_range, history['train_f1'], label='train_f1')
plt.plot(epochs_range, history['val_f1'], label='val_f1')
plt.legend()
plt.title('F1')
plt.show()

## 9) Grad-CAM de ejemplo (visualización)

In [ ]:
# Generar Grad-CAM sobre una muestra positiva (label=1)
try:
    generate_gradcam_visualization(model, test_dataset, device, target_label=1)
except Exception as e:
    print('Grad-CAM skipped:', e)

## 10) Notas
- Este notebook es una plantilla base para entrenar otros modelos: reemplaza la llamada a `build_resnet18` por otras arquitecturas (ej. `build_efficientnet`, `build_densenet`) y ajusta `CONFIG`.
- Para entrenamiento a gran escala añade checkpointing por época y logging (TensorBoard, Weights & Biases, etc.).